[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/constitutive/constitutive.ipynb)

# Electromagnetic Constitutive Maps

This example computes how linear media respond to the electromagnetic field and how light waves travel through isotropic glass, a birefringent crystal, a magnetic ferrite, and moving glass. Each material is modelled as a constitutive map from the electromagnetic field to the excitation it induces, so differences in permittivity, permeability, and observer motion alone determine the allowed wave speeds and polarizations.

Spacetime models the electromagnetic field as a bivector of swept electric planes and spatial magnetic planes, and the material excitation as an antibivector. A rest-frame medium sums permittivity dyads over `axes & time` with inverse-permeability dyads over dual spatial patches, Lorentz motion carries that map via `boost >> medium(boost << Field)`, and both source-free Maxwell equations pack without metric contraction into `(wavevectors ^ Field) + (wavevectors ^ media).dual()`.

In component tensor notation, a general linear medium couples field and excitation through a rank-4 constitutive tensor $G^{\mu\nu} = \tfrac12 \chi^{\mu\nu\rho\sigma} F_{\rho\sigma}$ (or split $3 \times 3$ matrices $\mathbf{D} = \boldsymbol{\varepsilon}\mathbf{E} + \boldsymbol{\xi}\mathbf{B}$ and $\mathbf{H} = \boldsymbol{\zeta}\mathbf{E} + \boldsymbol{\mu}^{-1}\mathbf{B}$), where moving media acquire explicit cross-coupling matrices $\boldsymbol{\xi}, \boldsymbol{\zeta}$ from $\mathbf{v} \times \mathbf{B}$ and $\mathbf{v} \times \mathbf{E}$ and an axion coupling enters through the Levi-Civita symbol $\theta\,\epsilon^{\mu\nu\rho\sigma} F_{\rho\sigma}$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA
from examples.electromagnetism.constitutive import render, scenarios
from examples.util.animation import save_figure, show_states

np.set_printoptions(precision=4, suppress=True)

# Spacetime algebra with spatial axes positive (`x+y+z+t-`).
mv = NumpyContext(STA).multivector
Scalar = STA.gatype.scalar()                                                    # Scalar+
Covector = STA.gatype.vector()                                                  # Vector+
Field = STA.gatype.bivector()                                                   # Bivector+
Excitation = STA.gatype.antibivector()                                          # Antibivector-
Direction = STA.gatype.antivector()                                             # Antivector-
Constitutive = STA.gatype((Excitation, Field))                                  # Antibivector- <- Bivector+
AxionCoupling = STA.gatype.antipseudoscalar()                                   # Scalar-
StressEnergy = STA.gatype((Direction, Direction))                               # Antivector- <- Antivector-
WaveMap = STA.gatype((STA.gatype.odd(), Field))                                 # Odd+ <- Bivector+

t, x, z = mv.t, mv.x, mv.z                                                      # Vector+, Vector+, Vector+
# The observer's direction in time pairs to one with t because t squares to minus one.
time = -t.dual()                                                                # [] Antivector-
axes = stack((x.dual(), mv.y.dual(), z.dual()))                                 # [3] Antivector-
magnetic_planes = stack((mv.yz, mv.zx, mv.xy))                                  # [3] Bivector+
swept_planes = axes & time                                                      # [3] Antibivector-
patches = magnetic_planes.dual()                                                # [3] Antibivector-
# Time squares to minus one, so electric dyads enter with a minus sign.
electric_dyads = -swept_planes * (Field & swept_planes)                         # [3] Antibivector- <- Bivector+
magnetic_dyads = patches * (Field & patches)                                    # [3] Antibivector- <- Bivector+
vacuum = Field.dual()                                                           # [] Antibivector- <- Bivector+
omega = 1.0

## Rest-frame media

Vacuum maps each field bivector to its dual antibivector. A material at rest weights each part of the field that the observer distinguishes with a dyad formed from an intrinsic readout patch times its pairing with the field. Swept spacetime planes `axes & time` read the electric components and carry the permittivities, while dual spatial patches `magnetic_planes.dual()` read the magnetic components and carry the inverse permeabilities. Isotropic glass weights all three dyads of each kind equally, a uniaxial crystal splits the electric weights, and a magnetic ferrite splits the magnetic weights.

In [ ]:
permittivities = scenarios.permittivities
inverse_permeabilities = scenarios.inverse_permeabilities
n_media = scenarios.n_media

In [ ]:
def medium(permittivities: np.ndarray, inverse_permeabilities: np.ndarray) -> Constitutive:
    """Weighted electric and magnetic dyads sum to the rest-frame constitutive map (`[...] Antibivector- <- Bivector+`) for permittivities and inverse permeabilities (`[..., 3]`)."""
    return (permittivities * electric_dyads + inverse_permeabilities * magnetic_dyads).sum(axis=-1)


media = medium(permittivities, inverse_permeabilities)                          # [n_media] Antibivector- <- Bivector+
glass, crystal, ferrite = media

## Energy and momentum currents

Pairing an excitation with a field forms the constitutive bilinear form on fields, giving minus twice the Lagrangian when evaluated on one field with itself. Together, the field and its excitation define the Minkowski stress-energy map from spacetime directions to energy-momentum currents without referencing the metric again, because the interior readout along a direction is simply `Direction & field`. Pairing the current along the observer's time direction with the spatial slice `t` gives the measured energy density.

In [ ]:
def stress_energy(field: Field, excitation: Excitation) -> StressEnergy:
    """The Minkowski stress-energy map (`[...] Antivector- <- Antivector-`) sends each spacetime direction to the energy-momentum current of `field` and `excitation`."""
    negative_lagrangian = (field & excitation) / 2                              # [...] Scalar+
    return negative_lagrangian * Direction - ((Direction & field) ^ excitation)


constitutive_form = Field & vacuum                                              # [] Scalar+ <- (Bivector+, Bivector+)

# A vacuum plane wave travelling along z cancels electric and magnetic parts in the Lagrangian.
vacuum_wave = (t - z) ^ x                                                       # [] Bivector+
vacuum_excitation = vacuum(vacuum_wave)                                         # [] Antibivector-
negative_lagrangian = constitutive_form(vacuum_wave, vacuum_wave) / 2           # [] Scalar+

stress_map = stress_energy(vacuum_wave, vacuum_excitation)                      # [] Antivector- <- Antivector-
stress_form = Covector & stress_map                                             # [] Scalar+ <- (Vector+, Antivector-)
energy_density = stress_form(t, time)                                           # [] Scalar+

# Reflecting across t reverses electric planes and keeps magnetic planes, turning the Lagrangian into energy.
energy_form = (t * Field * t.inverse()) & vacuum                                # [] Scalar+ <- (Bivector+, Bivector+)
field_energy_density = energy_form(vacuum_wave, vacuum_wave) / 2                # [] Scalar+

print("energy density:", energy_density.to_array(), field_energy_density.to_array())

For a plane wave in vacuum, equal electric and magnetic amplitudes cancel in the Lagrangian while adding to unit energy density under the observer's reflection. Inside matter, the stress-energy map yields the Minkowski current, whose bilinear form need not be symmetric when energy flux and momentum density differ.

In component tensor notation, the Minkowski stress-energy tensor reads as $T^{\mu}{}_{\nu} = F_{\nu\alpha} G^{\mu\alpha} - \tfrac14 \delta^{\mu}_{\nu} F_{\alpha\beta} G^{\alpha\beta}$ with observer energy density $u = T_{\mu\nu} u^\mu u^\nu$.

## Dispersion along an axis

A plane wave satisfies both source-free Maxwell equations along its spacetime wavevector, so wedging the wavevector with the field and with the excitation both vanish. Dualizing the excitation residual places the two exterior equations in grades 1 and 3 where they cannot cancel, so a propagating wave exists wherever the smallest singular value of the combined wave map drops to zero.

In differential-form notation, the two source-free equations read as $k \wedge F = 0$ and $k \wedge G = 0$.

In [ ]:
n_speeds = scenarios.n_speeds
speeds = np.linspace(*scenarios.speed_range, n_speeds)
threshold = scenarios.threshold
expected_speeds = scenarios.expected_speeds

In [ ]:
def wave_map(wavevectors: Covector, media: Constitutive) -> WaveMap:
    """The two exterior source-free Maxwell equations along `wavevectors` occupy separate odd grades (`[...] Odd+ <- Bivector+`)."""
    return (wavevectors ^ Field) + (wavevectors ^ media).dual()


def wave_residuals(wavevectors: Covector, media: Constitutive) -> Scalar:
    """The smallest singular value (`[...] Scalar+`) of the wave map vanishes along wavevectors that carry a plane wave."""
    return wave_map(wavevectors, media).svdvals()[..., -1]


wavevectors = speeds * t - z                                                    # [n_speeds] Vector+
residuals = wave_residuals(wavevectors[:, None], media)                         # [n_speeds, n_media] Scalar+
save_figure(render.dispersion_figure(speeds, residuals, expected_speeds), "constitutive_dispersion");

Each curve traces the smallest singular value of a medium's wave map across trial phase speeds along the propagation axis, plunging to zero on the dotted vertical lines that mark the analytic speeds. Isotropic glass dips once where its two transverse polarizations share a single speed, whereas the birefringent crystal and magnetic ferrite each split into two sharp V-shaped zeroes.

## Eigenmode polarizations and birefringence

At each phase speed where the residual vanishes, the trailing right singular vector of the wave map spans the field bivector that propagates without distortion. Isotropic glass carries two degenerate modes at a single speed, so any superposition preserves its polarization along the axis; the uniaxial crystal assigns distinct phase speeds to its two orthogonal modes, causing their relative phase and combined polarization to beat along the beam.

In [ ]:
n_samples = scenarios.n_samples
distances = scenarios.distances
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def local_minima(residual: Scalar, threshold: float) -> np.ndarray:
    """Interior speed samples (`[n_speeds - 2, ...]`) reach a wave mode where `residual` forms a local minimum below `threshold`."""
    left, middle, right = residual[:-2], residual[1:-1], residual[2:]
    return (middle <= left) & (middle <= right) & (middle < threshold)


def field_modes(wavevectors: Covector, media: Constitutive) -> Field:
    """The trailing right singular vector of the wave map spans the null-space wave field (`[...] Bivector+`)."""
    _, _, fields = wave_map(wavevectors, media).svd()
    return fields[..., -1]


crystal_speeds = speeds[1:-1][local_minima(residuals, threshold)[:, 1]]         # [2]
crystal_modes = crystal_speeds, field_modes(crystal_speeds * t - z, crystal)    # [2] Bivector+

(glass_phase_speed,) = speeds[1:-1][local_minima(residuals, threshold)[:, 0]]
_, _, glass_svd_fields = wave_map(glass_phase_speed * t - z, glass).svd()
glass_modes = np.full(2, glass_phase_speed), glass_svd_fields[-2:]              # [2] Bivector+

save_figure(render.polarizations_figure(crystal_modes), "constitutive_polarizations");

Crimson and blue arrows in the transverse plane mark the unit electric-field directions of the crystal's slow and fast eigenmodes along their dotted polarization lines.

In [ ]:
def travelling_fields(speeds: np.ndarray, fields: Field, distances: np.ndarray, n_frames: int) -> Iterator[Field]:
    """Yields the superposed travelling wave field (`[n_samples] Bivector+`) across `distances` over `n_frames` evenly spaced times of one period."""
    units = fields / (fields & time).norm()                                     # [2] Bivector+
    for tau in np.linspace(0.0, 2.0 * np.pi / omega, n_frames, endpoint=False):
        phases = np.cos(omega * (distances[:, None] / speeds - tau))            # [n_samples, 2]
        yield (units * phases).sum(axis=-1)                                     # [n_samples] Bivector+


save_figure(render.wave_comparison_figure(glass_modes, crystal_modes, distances), "constitutive_birefringence_3d");
state_pairs = zip(
    travelling_fields(*glass_modes, distances, n_frames),
    travelling_fields(*crystal_modes, distances, n_frames),
)
streamed_pairs = show_states(
    state_pairs,
    render.wave_comparison_frames,
    glass_modes,
    crystal_modes,
    distances,
    name="constitutive_comparison",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

Crimson arrows trace the electric field along the propagation rod while translucent blue discs lie in the transverse magnetic planes with rim chevrons showing their sense of circulation. On the left in isotropic glass, both modes travel together in a fixed plane; on the right in the birefringent crystal, the faster mode outruns the slower one so the superposed electric tip corkscrews between linear and elliptical polarization along the beam.

## Phase-speed surfaces by heading

Rotating the spatial wavevector across headings in a plane maps out the normal-speed surface of each medium. Isotropic glass at rest traces a centred circle, the uniaxial crystal splits into two sheets that kiss along the optic axis where both transverse polarizations see equal permittivity, and moving glass shifts its phase-speed surface forward along the direction of motion.

In [ ]:
n_headings = scenarios.n_headings
n_heading_speeds = scenarios.n_heading_speeds
heading_speeds = np.linspace(*scenarios.heading_speed_range, n_heading_speeds)
glass_speed = scenarios.glass_speed

In [ ]:
def moving_medium(rest_medium: Constitutive, fractions: np.ndarray) -> Constitutive:
    """Lorentz boosts along `z` at speed `fractions` carry `rest_medium` into the observer's frame (`[...] Antibivector- <- Bivector+`)."""
    boosts = ((z ^ t) * (np.arctanh(fractions) / 2)).exp()                      # [...] Even+
    return boosts >> rest_medium(boosts << Field)                               # [...] Antibivector- <- Bivector+


angles = np.linspace(0, 2 * np.pi, n_headings)
# Spatial wavevector covectors turned from z toward x across the xz plane.
slices = (mv.zx * (-angles / 2)).exp() >> z                                     # [n_headings] Vector+
heading_wavevectors = heading_speeds[:, None] * t - slices                      # [n_heading_speeds, n_headings] Vector+
heading_cases = stack((glass, crystal, moving_medium(glass, np.array(glass_speed))))  # [3] Antibivector- <- Bivector+

heading_residuals = wave_residuals(heading_wavevectors[..., None], heading_cases)  # [n_heading_speeds, n_headings, 3] Scalar+
save_figure(render.fresnel_surface_figure(angles, heading_residuals, heading_speeds, threshold), "constitutive_fresnel_surface");

Polar radius measures phase speed from the origin along each propagation heading between the vertical motion axis and the horizontal optic axis. Stationary glass forms the centred blue circle, the crystal's two orange sheets kiss along the horizontal optic axis, and moving glass shifts into the green oval dragged upward in the direction of motion.

## Fresnel drag in moving glass

A moving dielectric responds in its own rest frame, so sandwiching with a Lorentz boost `boosts >> rest_medium(boosts << Field)` pulls the observer's field into the medium's rest frame, applies the isotropic response there, and pushes the excitation back out. Co-propagating waves speed up and counter-propagating waves slow down according to exact relativistic velocity addition, departing at high speeds from Fresnel's linear drag approximation.

In one-dimensional relativistic kinematics, combining rest-frame phase speed $1/n$ with medium speed $\beta$ gives $v = (1/n \pm \beta)/(1 \pm \beta/n)$, whose first-order Taylor expansion in $\beta$ is $v \approx 1/n \pm (1 - 1/n^2)\beta$.

In [ ]:
n_drags = scenarios.n_drags
fractions = np.linspace(*scenarios.drag_range, n_drags)
n_drag_speeds = scenarios.n_drag_speeds
drag_speeds = np.linspace(*scenarios.drag_speed_range, n_drag_speeds)
senses = scenarios.senses
n_senses = scenarios.n_senses
n_curve = scenarios.n_curve

In [ ]:
def minimum_speeds(speeds: np.ndarray, residual: Scalar) -> np.ndarray:
    """The minimum of `residual` along the leading sample axis selects the phase speed (`[...]`) from `speeds`."""
    return speeds[residual.argmin(axis=0)]


moving_glass = moving_medium(glass, fractions)                                  # [n_drags] Antibivector- <- Bivector+
drag_wavevectors = drag_speeds[:, None, None] * t - senses[:, None] * z         # [n_drag_speeds, n_senses, 1] Vector+
found_speeds = minimum_speeds(drag_speeds, wave_residuals(drag_wavevectors, moving_glass))  # [n_senses, n_drags]

light = expected_speeds[0, 0]
curve = np.linspace(*scenarios.drag_range, n_curve)
flow = senses[:, None] * curve                                                  # [n_senses, n_curve]
relativistic = (light + flow) / (1 + light * flow)
first_order = light + flow * (1 - light ** 2)
save_figure(render.fresnel_drag_figure(fractions, found_speeds, curve, relativistic, first_order), "constitutive_fresnel_drag");

Red circles for waves travelling with the glass and blue squares for waves travelling against it lie directly on the solid relativistic curves and bend away from the dashed straight lines of first-order Fresnel drag as the medium approaches relativistic speed.

In [ ]:
# checks
speed_step = speeds[1] - speeds[0]
assert (negative_lagrangian.abs() < 1e-12).all()
assert ((energy_density - 1.0).abs() < 1e-12).all()
assert ((field_energy_density - 1.0).abs() < 1e-12).all()
for medium_minima, medium_speeds in zip(local_minima(residuals, threshold).T, expected_speeds):
    assert np.allclose(speeds[1:-1][medium_minima], np.unique(medium_speeds), atol=speed_step)
crystal_wavevectors = crystal_speeds * t - z                                    # [2] Vector+
for residual_term in (crystal_wavevectors ^ crystal_modes[1], crystal_wavevectors ^ crystal(crystal_modes[1])):
    assert (residual_term.reverse().scalar_product(t * residual_term.involute() * t.inverse()) < speed_step ** 2).all()
assert len(streamed_pairs) == n_frames
first_field = streamed_pairs[0][1]
assert (first_field.reverse().scalar_product(t * first_field.involute() * t.inverse()) > 0).any()
found_flow = senses[:, None] * fractions                                        # [n_senses, n_drags]
assert np.allclose(found_speeds, (light + found_flow) / (1 + light * found_flow), atol=drag_speeds[1] - drag_speeds[0])